In [0]:
from pyspark.sql import functions as F

CATALOG = "healthcare_medallion_dbw"

appointments_bronze = spark.table(
    f"{CATALOG}.bronze.appointments"
)

appointments_bronze.printSchema()

root
 |-- appointment_id: string (nullable = true)
 |-- patient_id: string (nullable = true)
 |-- doctor_id: string (nullable = true)
 |-- appointment_date: date (nullable = true)
 |-- appointment_time: timestamp (nullable = true)
 |-- reason_for_visit: string (nullable = true)
 |-- status: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _source_file_name: string (nullable = true)
 |-- _batch_id: string (nullable = true)
 |-- _layer: string (nullable = true)
 |-- _ingestion_date: date (nullable = true)
 |-- _pipeline_version: string (nullable = true)
 |-- _source_system: string (nullable = true)
 |-- _record_hash: string (nullable = true)
 |-- _is_duplicate: boolean (nullable = true)
 |-- _raw_row_number: long (nullable = true)



In [0]:
appointments_silver = (
    appointments_bronze

    .dropDuplicates(["appointment_id"])

    .withColumn(
        "patient_id",
        F.trim(F.col("patient_id"))
    )

    .withColumn(
        "doctor_id",
        F.trim(F.col("doctor_id"))
    )

    .withColumn(
        "reason_for_visit",
        F.initcap(F.trim(F.col("reason_for_visit")))
    )

    .withColumn(
        "status",
        F.upper(F.trim(F.col("status")))
    )

    .withColumn(
        "no_show_flag",
        F.when(
            F.col("status") == "NO-SHOW",
            1
        ).otherwise(0)
    )

    .withColumn(
        "completed_flag",
        F.when(
            F.col("status") == "COMPLETED",
            1
        ).otherwise(0)
    )

    .withColumn(
        "cancelled_flag",
        F.when(
            F.col("status") == "CANCELLED",
            1
        ).otherwise(0)
    )

    .withColumn(
        "_dq_passed",
        (
            F.col("appointment_id").isNotNull()
            &
            F.col("patient_id").isNotNull()
            &
            F.col("doctor_id").isNotNull()
            &
            F.col("appointment_date").isNotNull()
        )
    )

    .withColumn(
        "_dq_score",
        (
            F.when(F.col("appointment_id").isNotNull(), 1).otherwise(0)
            +
            F.when(F.col("patient_id").isNotNull(), 1).otherwise(0)
            +
            F.when(F.col("doctor_id").isNotNull(), 1).otherwise(0)
            +
            F.when(F.col("appointment_date").isNotNull(), 1).otherwise(0)
        ) / F.lit(4.0)
    )

    .withColumn(
        "_dq_failure_reason",
        F.when(
            F.col("_dq_passed") == False,
            F.lit("Mandatory appointment field missing")
        )
    )

    .withColumn(
        "_silver_load_timestamp",
        F.current_timestamp()
    )

    .withColumn(
        "_silver_batch_id",
        F.expr("uuid()")
    )

    .withColumn(
        "_bronze_batch_id",
        F.col("_batch_id")
    )

    .withColumn(
        "_enrichment_source",
        F.lit("bronze.appointments")
    )
)

In [0]:
patients_keys = (
    spark.table(
        f"{CATALOG}.silver.patients"
    )
    .select("patient_id")
    .dropDuplicates()
)

doctors_keys = (
    spark.table(
        f"{CATALOG}.silver.doctors"
    )
    .select("doctor_id")
    .dropDuplicates()
)

appointments_silver = (
    appointments_silver

    .join(
        patients_keys.withColumn(
            "valid_patient",
            F.lit(True)
        ),
        on="patient_id",
        how="left"
    )

    .join(
        doctors_keys.withColumn(
            "valid_doctor",
            F.lit(True)
        ),
        on="doctor_id",
        how="left"
    )

    .withColumn(
        "valid_patient",
        F.coalesce(
            F.col("valid_patient"),
            F.lit(False)
        )
    )

    .withColumn(
        "valid_doctor",
        F.coalesce(
            F.col("valid_doctor"),
            F.lit(False)
        )
    )

    .withColumn(
        "_dq_passed",
        F.col("_dq_passed")
        &
        F.col("valid_patient")
        &
        F.col("valid_doctor")
    )
)

In [0]:
(
    appointments_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{CATALOG}.silver.appointments"
    )
)

In [0]:
print(
    "Silver appointments:",
    spark.table(
        f"{CATALOG}.silver.appointments"
    ).count()
)

display(
    spark.table(
        f"{CATALOG}.silver.appointments"
    )
)

Silver appointments: 200


doctor_id,patient_id,appointment_id,appointment_date,appointment_time,reason_for_visit,status,_ingestion_timestamp,_source_file_name,_batch_id,_layer,_ingestion_date,_pipeline_version,_source_system,_record_hash,_is_duplicate,_raw_row_number,no_show_flag,completed_flag,cancelled_flag,_dq_passed,_dq_score,_dq_failure_reason,_silver_load_timestamp,_silver_batch_id,_bronze_batch_id,_enrichment_source,valid_patient,valid_doctor
D001,P031,A165,2023-04-04,2026-08-10T15:30:00.000Z,Consultation,CANCELLED,2026-08-10T15:37:48.067Z,abfss://landing@tanvihealthstore2608.dfs.core.windows.net/appointments/appointments.csv,f85645bc-e96c-4b26-a616-4a8fc95a69fa,BRONZE,2026-08-10,1.0,appointments,09f727358dfc49ff582284d538c349fb60cf4556dca68066b16442a8315e66c8,false,5,0,0,1,true,1.0,null,2026-08-10T15:59:28.360Z,cebc17a2-0107-47c0-b62b-5d96a3e83842,f85645bc-e96c-4b26-a616-4a8fc95a69fa,bronze.appointments,true,true
D003,P016,A159,2023-04-08,2026-08-10T16:15:00.000Z,Emergency,NO-SHOW,2026-08-10T15:37:48.067Z,abfss://landing@tanvihealthstore2608.dfs.core.windows.net/appointments/appointments.csv,f85645bc-e96c-4b26-a616-4a8fc95a69fa,BRONZE,2026-08-10,1.0,appointments,3c693f4040f4f9bf273509922c7c92a938dd003fb911b577bbecf1af75761ac9,false,34,1,0,0,true,1.0,null,2026-08-10T15:59:28.360Z,b0a5303b-5489-4bca-93fb-9fab56d4a93a,f85645bc-e96c-4b26-a616-4a8fc95a69fa,bronze.appointments,true,true
D007,P012,A143,2023-09-21,2026-08-10T12:15:00.000Z,Checkup,CANCELLED,2026-08-10T15:37:48.067Z,abfss://landing@tanvihealthstore2608.dfs.core.windows.net/appointments/appointments.csv,f85645bc-e96c-4b26-a616-4a8fc95a69fa,BRONZE,2026-08-10,1.0,appointments,477e5d493a333f9ebf23bed023622d6e10640c84560432b98763327ffaaff75d,false,47,0,0,1,true,1.0,null,2026-08-10T15:59:28.360Z,dc02256f-f8bb-49bc-9eeb-02cc61723119,f85645bc-e96c-4b26-a616-4a8fc95a69fa,bronze.appointments,true,true
D008,P045,A050,2023-08-16,2026-08-10T15:00:00.000Z,Consultation,NO-SHOW,2026-08-10T15:37:48.067Z,abfss://landing@tanvihealthstore2608.dfs.core.windows.net/appointments/appointments.csv,f85645bc-e96c-4b26-a616-4a8fc95a69fa,BRONZE,2026-08-10,1.0,appointments,512084a41d59b7a7b954c6c6ce2ed82cdc1d38da6f75b473af61f249695c80f9,false,57,1,0,0,true,1.0,null,2026-08-10T15:59:28.360Z,3c538c97-a94c-4e1a-9b8d-fd0ef948306b,f85645bc-e96c-4b26-a616-4a8fc95a69fa,bronze.appointments,true,true
D003,P029,A012,2023-05-07,2026-08-10T10:00:00.000Z,Follow-up,COMPLETED,2026-08-10T15:37:48.067Z,abfss://landing@tanvihealthstore2608.dfs.core.windows.net/appointments/appointments.csv,f85645bc-e96c-4b26-a616-4a8fc95a69fa,BRONZE,2026-08-10,1.0,appointments,5c893390617d0466f31860979cb76a4caf4411d537204fdce6e534b78c9beeaf,false,68,0,1,0,true,1.0,null,2026-08-10T15:59:28.360Z,3448c547-2c02-427e-ad1b-6331e4cf16b0,f85645bc-e96c-4b26-a616-4a8fc95a69fa,bronze.appointments,true,true
D007,P032,A047,2023-05-02,2026-08-10T11:00:00.000Z,Therapy,COMPLETED,2026-08-10T15:37:48.067Z,abfss://landing@tanvihealthstore2608.dfs.core.windows.net/appointments/appointments.csv,f85645bc-e96c-4b26-a616-4a8fc95a69fa,BRONZE,2026-08-10,1.0,appointments,5eeef6383d44774d43a82fa11698489d3164f46815af7e011799374b1e951674,false,70,0,1,0,true,1.0,null,2026-08-10T15:59:28.360Z,68688a89-7f10-4f56-bd96-2598333c48d2,f85645bc-e96c-4b26-a616-4a8fc95a69fa,bronze.appointments,true,true
D006,P022,A198,2023-05-15,2026-08-10T08:30:00.000Z,Therapy,NO-SHOW,2026-08-10T15:37:48.067Z,abfss://landing@tanvihealthstore2608.dfs.core.windows.net/appointments/appointments.csv,f85645bc-e96c-4b26-a616-4a8fc95a69fa,BRONZE,2026-08-10,1.0,appointments,6424767e2183fcf994b2963912405b197efdba9273f7ba8e6c104b376b6de1fc,false,73,1,0,0,true,1.0,null,2026-08-10T15:59:28.360Z,aa2cb79f-e107-40f1-9ec4-4e73754996c8,f85645bc-e96c-4b26-a616-4a8fc95a69fa,bronze.appointments,true,true
D008,P013,A124,2023-03-16,2026-08-10T17:15:00.000Z,Emergency,CANCELLED,2026-08-10T15:37:48.067Z,abfss://landing@tanvihealthstore2608.dfs.core.windows.net/appointments/appointments.csv,f85645bc-e96c-4b26-a616-4a8fc95a69fa,BRONZ

In [0]:
spark.table(
    f"{CATALOG}.silver.appointments"
).groupBy("status").count().show()

+---------+-----+
|   status|count|
+---------+-----+
|CANCELLED|   51|
|  NO-SHOW|   52|
|COMPLETED|   46|
|SCHEDULED|   51|
+---------+-----+

